Excludes any sequence that was part of training.
Merges the remaining sequences with my R-generated shapes.
Aligns columns to the exact feature list the model was trained on.
Predicts and computes an uncertainty score (tree-std).



to check how far away the sequnces are away of the training datat, the 8 selcetesd sequcnes during multiple rouend of random genaration of sequnces and selection from the best models outputs. All the sequnces used are 5 hummingbird distances away form the training data.

In [20]:
import pandas as pd

# ---------- INPUT FILES ----------
LIB_FILE = "Up_library.tsv"          # original training dataset
SELECTED_FILE = "sequences.csv"       # your chosen sequences

# ---------- LOAD DATA ----------
lib_df = pd.read_csv(LIB_FILE)
sel_df = pd.read_csv(SELECTED_FILE)

# assumes column name is "sequence"
# first column from TSV (unnamed)
library = lib_df.iloc[:, 0].astype(str).str.upper().tolist()

# named column from CSV
selected = sel_df["Sequence"].astype(str).str.upper().tolist()

# ---------- HAMMING DISTANCE ----------
def hamming_distance(s1, s2):
    return sum(a != b for a, b in zip(s1, s2))

# ---------- FIND CLOSEST TRAINING SEQUENCE ----------
results = []

for seq in selected:
    
    best_dist = 19
    best_match = None
    
    for train_seq in library:
        
        d = hamming_distance(seq, train_seq)
        
        if d < best_dist:
            best_dist = d
            best_match = train_seq
    
    results.append({
        "selected_sequence": seq,
        "closest_training_sequence": best_match,
        "min_hamming_distance": best_dist
    })

# ---------- SAVE RESULTS ----------
results_df = pd.DataFrame(results)

print(results_df)

results_df.to_csv("selected_vs_training_hamming.csv", index=False)

print("\nSaved: selected_vs_training_hamming.csv")

     selected_sequence                          closest_training_sequence  \
0  CTGGGTGTGAAAAGGGCTT  CTGAATGTCAATATGGCTT\t1.419034705\t1.584701479\...   
1  GGAAAACGGCTGATGGGGA  GAGAAATGCCTGATGAGTA\t1.876071497\t2.198993741\...   
2  GAAACAGGGTTGCACATTG  CACAGAGGGTTGCGCATCA\t1.944258438\t1.834583077\...   
3  ACTAGGTCTAGACACAAAG  ATTATGTCTAGATACGATT\t1.992779107\t1.834583077\...   
4  TAAGGCTCAGGTATAAAAG  TAAACCTCAGGTATGGAAT\t2.022069362\t2.067805902\...   
5  CACGGATAGGTTGTAAAAG  AAGTGCTAGGGTGTAAAAG\t2.659563142\t2.875756403\...   
6  GTTAAAAATTTTGCAATTA  CTTAATAACTAGGCAATTA\t1.980298181\t1.775087459\...   
7  GGGCTACACTCTCGCCTTC  TGAGTAGACTCTCTCCTTT\t1.91524608\t1.834583077\t...   

   min_hamming_distance  
0                     5  
1                     6  
2                     6  
3                     6  
4                     5  
5                     5  
6                     5  
7                     6  

Saved: selected_vs_training_hamming.csv


Getting 100000 sequnces not seen by the modele

In [21]:
# This cell generates a synthetic `unseen_pool.csv` with a big pool of 19-mers
# and a minimal empty `merged_output.csv` so your script can run as-is.
# The pool is filtered to avoid extreme GC and long homopolymers, and
# is enriched so each special bucket has plenty of candidates.

import random, csv, os, math
from collections import Counter

random.seed(42)

OUT_DIR = "./outputs"     # relative to your current working directory

POOL_PATH = os.path.join(OUT_DIR, "unseen_pool.csv")
TRAIN_PATH = os.path.join(OUT_DIR, "merged_output_13infile.csv")

N_RANDOM = 50000           # base randoms
GC_MIN, GC_MAX = 0.35, 0.65
MAX_HOMOPOLYMER = 4

POS20, POS21, POS22, POS23, POS24 = 15, 16, 17, 18, 19  # 1-indexed
BLOCKS = ["AATAA","ATATA","GCGCG","CGCGC"]
ENRICH_PER_BLOCK = 200      # ensure at least this many for each special block
ENRICH_POS_RULES = {
    "pos22=T": ("T", POS22),
    "pos22=G": ("G", POS22),
    "pos21=A": ("A", POS21),
    "pos20=G": ("G", POS20),
}
ENRICH_PER_POS = 200        # for each positional bucket

alphabet = "ACGT"

def max_homopolymer(s: str) -> int:
    run = best = 1
    for i in range(1, len(s)):
        if s[i] == s[i-1]:
            run += 1
            best = max(best, run)
        else:
            run = 1
    return best

def ok_seq(s: str) -> bool:
    gc = (s.count("G")+s.count("C"))/19.0
    return (GC_MIN <= gc <= GC_MAX) and (max_homopolymer(s) <= MAX_HOMOPOLYMER)

def rand_seq(n=19) -> str:
    return "".join(random.choice(alphabet) for _ in range(n))

seqs = set()

# Base random pool with filtering
while len(seqs) < N_RANDOM:
    s = rand_seq(19)
    if ok_seq(s):
        seqs.add(s)

# Enrich for block patterns at positions 20–24 (i.e., indices 14..19)
for block in BLOCKS:
    need = ENRICH_PER_BLOCK
    tries = 0
    while need > 0 and tries < ENRICH_PER_BLOCK * 100:
        tries += 1
        s_list = [random.choice(alphabet) for _ in range(19)]
        # impose block at positions 15..19 (1-indexed)
        s_list[POS20-1:POS24] = list(block)
        s = "".join(s_list)
        if ok_seq(s):
            if s not in seqs:
                seqs.add(s)
                need -= 1

# Enrich for single-position buckets
for name, (base, pos1) in ENRICH_POS_RULES.items():
    need = ENRICH_PER_POS
    tries = 0
    while need > 0 and tries < ENRICH_PER_POS * 100:
        tries += 1
        s_list = [random.choice(alphabet) for _ in range(19)]
        s_list[pos1-1] = base
        s = "".join(s_list)
        if ok_seq(s):
            if s not in seqs:
                seqs.add(s)
                need -= 1

# Write unseen_pool.csv
os.makedirs(OUT_DIR, exist_ok=True)
with open(POOL_PATH, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["sequence"])
    for s in seqs:
        w.writerow([s])

# Write an empty merged_output.csv with a 'sequence' header (no rows)
with open(TRAIN_PATH, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["sequence"])

from collections import defaultdict
# Quick bucket counts (for your sanity check)
bcounts = defaultdict(int)
for s in seqs:
    if s[POS22-1] == "T": bcounts["pos22=T"] += 1
    if s[POS22-1] == "G": bcounts["pos22=G"] += 1
    if s[POS21-1] == "A": bcounts["pos21=A"] += 1
    if s[POS20-1] == "G": bcounts["pos20=G"] += 1
    block = s[POS20-1:POS24]
    if block in BLOCKS: bcounts[f"{block}@20–24"] += 1

len(seqs), dict(bcounts), POOL_PATH, TRAIN_PATH


(51600,
 {'pos21=A': 13151,
  'pos22=T': 12958,
  'pos22=G': 12952,
  'pos20=G': 13273,
  'CGCGC@20–24': 234,
  'ATATA@20–24': 231,
  'GCGCG@20–24': 235,
  'AATAA@20–24': 233},
 './outputs/unseen_pool.csv',
 './outputs/merged_output_13infile.csv')

Filtering the sequnces after the hyphotesis and the distance

In [22]:
import os, re, numpy as np, pandas as pd

# ---- CONFIG: point these to your files ----
POOL_CSV   = "unseen_pool.csv"      # 19-mers with a 'sequence' column (or first column)
TRAIN_FILE = "merged_output_13infile.csv"    # 19- or 27-mers with a 'sequence' column (or first column)
OUT_DIR    = "outputs"              # will be created if missing
OUT_BASENAME = "panel20"            # files: panel20_candidates.csv / panel20_features.csv
TARGET_TOTAL = 20                   # how many sequences to export
MIN_HAMMING  = 3                    # novelty vs training (raise to 4 if you still get many)
PER_BUCKET_CAP = 5                  # max from any single bucket
# ------------------------------------------

POS20, POS21, POS22, POS23, POS24 = 15, 16, 17, 18, 19
DNA_ONLY = re.compile(r"[^ACGT]")
_nt2u8 = {ord('A'):0, ord('C'):1, ord('G'):2, ord('T'):3}
_u8to = np.array(list("ACGT"), dtype="U1")

def _load_19(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.upper().str.replace(DNA_ONLY, "", regex=True)
    return s[s.str.len()==19]

def load_pool(path: str) -> list[str]:
    df = pd.read_csv(path, sep=None, engine="python")
    col = "sequence" if "sequence" in df.columns else df.columns[0]
    return _load_19(df[col]).tolist()

def load_train_cores(path: str) -> list[str]:
    df = pd.read_csv(path, sep=None, engine="python")
    col = "sequence" if "sequence" in df.columns else df.columns[0]
    s = df[col].astype(str).str.upper().str.replace(DNA_ONLY, "", regex=True)
    s = s[(s.str.len()==19) | (s.str.len()==27)]
    cores = s.where(s.str.len()==19, s.str.slice(4, -4))  # middle 19 if 27-mer
    return cores.tolist()

def encode_u8(seqs: list[str]) -> np.ndarray:
    arr = np.frombuffer("".join(seqs).encode("ascii"), dtype=np.uint8).reshape(len(seqs), -1)
    out = np.empty_like(arr)
    vmap = np.zeros(256, dtype=np.uint8) + 255
    for k,v in _nt2u8.items(): vmap[k] = v
    np.take(vmap, arr, out=out)
    if np.any(out==255): raise ValueError("Non-ACGT found.")
    return out

def hamming_min_to_lib(cand_u8: np.ndarray, lib_u8: np.ndarray, block: int = 200_000) -> int:
    mn = 19
    n  = lib_u8.shape[0]
    for i in range(0, n, block):
        d = np.sum(lib_u8[i:i+block] != cand_u8, axis=1, dtype=np.int16)
        m = int(d.min())
        if m < mn:
            mn = m
            if mn == 0: return 0
    return mn

def mine_buckets(pool: list[str]) -> dict[str, list[str]]:
    buckets = {
        "pos22=T": [], "pos22=G": [], "pos21=A": [], "pos20=G": [],
        "AATAA@20–24": [], "ATATA@20–24": [], "GCGCG@20–24": [], "CGCGC@20–24": [],
    }
    for s in pool:
        if s[POS22-1] == "T": buckets["pos22=T"].append(s)
        if s[POS22-1] == "G": buckets["pos22=G"].append(s)
        if s[POS21-1] == "A": buckets["pos21=A"].append(s)
        if s[POS20-1] == "G": buckets["pos20=G"].append(s)
        block = s[POS20-1:POS24]
        if block in ("AATAA","ATATA","GCGCG","CGCGC"):
            buckets[f"{block}@20–24"].append(s)
    return buckets

def select_panel(pool: list[str], train_u8: np.ndarray,
                 target_total=20, min_hamming=3, per_bucket_cap=5) -> list[str]:
    buckets = mine_buckets(pool)
    order = ["pos22=T","pos21=A","pos20=G","pos22=G",
             "AATAA@20–24","ATATA@20–24","GCGCG@20–24","CGCGC@20–24"]
    kept, seen = [], set()
    # bucket-balanced pass
    for k in order:
        group = buckets.get(k, [])
        taken_here = 0
        for s in group:
            if taken_here >= per_bucket_cap: break
            if s in seen: continue
            cu8 = encode_u8([s])[0]
            if hamming_min_to_lib(cu8, train_u8) >= min_hamming:
                kept.append(s); seen.add(s); taken_here += 1
                if len(kept) >= target_total: break
        if len(kept) >= target_total: break
    # top up if short
    if len(kept) < target_total:
        for s in pool:
            if s in seen: continue
            cu8 = encode_u8([s])[0]
            if hamming_min_to_lib(cu8, train_u8) >= min_hamming:
                kept.append(s); seen.add(s)
                if len(kept) >= target_total: break
    return kept

def max_homopolymer(s: str) -> int:
    run = best = 1
    for i in range(1, len(s)):
        if s[i]==s[i-1]:
            run+=1; best=max(best, run)
        else:
            run=1
    return best

def dinuc_freqs(s: str) -> dict[str, float]:
    counts = {a+b:0 for a in "ACGT" for b in "ACGT"}
    for i in range(len(s)-1):
        counts[s[i:i+2]] += 1
    total = len(s)-1
    return {k: v/total for k,v in counts.items()}

def make_feature_row(s: str) -> dict:
    row = {"sequence": s}
    row["GC_frac"] = (s.count("G")+s.count("C"))/19.0
    row["A_count"] = s.count("A"); row["C_count"] = s.count("C")
    row["G_count"] = s.count("G"); row["T_count"] = s.count("T")
    row["max_homopolymer"] = max_homopolymer(s)
    row["pos20"] = s[POS20-1]; row["pos21"] = s[POS21-1]; row["pos22"] = s[POS22-1]
    row["pos23"] = s[POS23-1]; row["pos24"] = s[POS24-1]
    row["block20_24"] = s[POS20-1:POS24]
    row.update({f"di_{k}": v for k,v in dinuc_freqs(s).items()})
    return row

# ---- run ----
pool = load_pool(POOL_CSV)
train_cores = load_train_cores(TRAIN_FILE)
train_u8 = encode_u8(train_cores)

panel_seqs = select_panel(pool, train_u8,
                          target_total=TARGET_TOTAL,
                          min_hamming=MIN_HAMMING,
                          per_bucket_cap=PER_BUCKET_CAP)

panel_df = pd.DataFrame({"sequence": panel_seqs})
features_df = pd.DataFrame([make_feature_row(s) for s in panel_seqs])

os.makedirs(OUT_DIR, exist_ok=True)
panel_path    = os.path.join(OUT_DIR, f"{OUT_BASENAME}_candidates.csv")
features_path = os.path.join(OUT_DIR, f"{OUT_BASENAME}_features.csv")
panel_df.to_csv(panel_path, index=False)
features_df.to_csv(features_path, index=False)

print(f"Saved {len(panel_df)} candidates to: {panel_path}")
print(f"Saved features table to:            {features_path}")
print("\nPreview (candidates):")
print(panel_df.head(10).to_string(index=False))
print("\nPreview (features):")
print(features_df.head(5).to_string(index=False))


Saved 20 candidates to: outputs/panel20_candidates.csv
Saved features table to:            outputs/panel20_features.csv

Preview (candidates):
           sequence
CCGGTTCCACTATATGTTG
GTTGATACGGTAAATGTGC
ACTTCCGGTCCTCGATTTA
AGCTATCTGTCCGTGGTGT
CCGCCGAGTATCGTAATCG
CTCTTAAAGACACGCAGCT
TCCGAGTACGTGCTTATGA
ACGCCAAGCCACGTAATAA
AGACACCGTCGGGCAAACT
TGGTTGTGAGTTGCTAGTC

Preview (features):
           sequence  GC_frac  A_count  C_count  G_count  T_count  max_homopolymer pos20 pos21 pos22 pos23 pos24 block20_24    di_AA    di_AC    di_AG    di_AT    di_CA    di_CC    di_CG    di_CT    di_GA    di_GC    di_GG    di_GT    di_TA    di_TC    di_TG    di_TT
CCGGTTCCACTATATGTTG 0.473684        3        5        4        7                2     T     G     T     T     G      TGTTG 0.000000 0.055556 0.000000 0.111111 0.055556 0.111111 0.055556 0.055556 0.000000 0.000000 0.055556 0.111111 0.111111 0.055556 0.111111 0.111111
GTTGATACGGTAAATGTGC 0.421053        5        2        6        6                3 

sorting the sequnces further

In [23]:
import pandas as pd
import numpy as np
import re

# === CONFIG ===
# If you already saved the 20 sequences to a CSV from your previous step,
# point to it here (must have a column named 'sequence').
IN_CSV  = "outputs/panel20_candidates.csv"       # <-- change if your filename is different
OUT_CSV = "panel20_labeled.csv"

# If you don't have IN_CSV, you can paste sequences here instead:
# (comment this block out if you're reading from IN_CSV)
seqs_manual = [
    # paste your 20 sequences as 19-mers, uppercase A/C/G/T, one per line:
    # "CAGGTCATGGACTTCTTCT",
    # "CAGGATATATTTGCGCTGC",
    # ...
]

# === UTIL ===
POS20, POS21, POS22, POS23, POS24 = 15, 16, 17, 18, 19  # 1-based inside the 19-bp UP window
DNA_ONLY = re.compile(r"[^ACGT]")

def clean_19(s):
    s = re.sub(DNA_ONLY, "", s.upper())
    if len(s) != 19:
        raise ValueError(f"Sequence length not 19: {s} ({len(s)})")
    return s

def gc_frac(s):
    return (s.count("G") + s.count("C")) / 19.0

def block_20_24(s):
    return s[POS20-1:POS24]  # 5-mer

def max_homopolymer(s):
    run = best = 1
    for i in range(1, len(s)):
        if s[i]==s[i-1]:
            run += 1
            best = max(best, run)
        else:
            run = 1
    return best

def label_sequence(s):
    """
    Returns:
      bucket_list (list of strings),
      predicted ('increase'/'decrease'/'mixed'/'neutral'),
      rationale (short text)
    Uses your SHAP-driven rules:
      - pos22: T → boost, G → reduce
      - pos21: A → boost
      - pos20: G → reduce
      - block 20–24: AT-rich (AATAA/ATATA/AAAAA/TTTTT) → boost; GC-rich (GCGCG/CGCGC/GGGGG/CCCCC) → reduce
      - very high/very low GC (heuristic) → reduce
    """
    b = []
    effects = []   # list of +1 (increase), -1 (decrease)

    # positions
    if s[POS22-1] == "T": b.append("pos22=T"); effects.append(+1)
    if s[POS22-1] == "G": b.append("pos22=G"); effects.append(-1)
    if s[POS21-1] == "A": b.append("pos21=A"); effects.append(+1)
    if s[POS20-1] == "G": b.append("pos20=G"); effects.append(-1)

    # shape-ish 5-mer at 20–24
    blk = block_20_24(s)
    if blk in ("AATAA", "ATATA", "AAAAA", "TTTTT"):
        b.append(f"{blk}@20–24"); effects.append(+1)
    if blk in ("GCGCG", "CGCGC", "GGGGG", "CCCCC"):
        b.append(f"{blk}@20–24"); effects.append(-1)

    # GC heuristic (very low/high can hurt)
    gcf = gc_frac(s)
    if gcf <= 0.32:
        b.append("GC%<=0.32"); effects.append(-1)
    elif gcf >= 0.68:
        b.append("GC%>=0.68"); effects.append(-1)

    # summarize predicted direction
    if not effects:
        predicted = "neutral"
    else:
        score = np.sign(sum(effects))
        predicted = {1:"increase", -1:"decrease", 0:"mixed"}[score]

    rationale = "; ".join(b) if b else "no strong rule-hit"
    return b, predicted, rationale

# === LOAD ===
if len(seqs_manual) > 0:
    seqs = [clean_19(s) for s in seqs_manual]
    df_in = pd.DataFrame({"sequence": seqs})
else:
    df_in = pd.read_csv(IN_CSV)
    col = "sequence" if "sequence" in df_in.columns else df_in.columns[0]
    df_in = df_in[[col]].rename(columns={col: "sequence"})
    df_in["sequence"] = df_in["sequence"].astype(str).map(clean_19)

# === LABEL ===
rows = []
for s in df_in["sequence"]:
    buckets, predicted, rationale = label_sequence(s)
    rows.append({
        "sequence": s,
        "GC_frac": round(gc_frac(s), 6),
        "pos20": s[POS20-1],
        "pos21": s[POS21-1],
        "pos22": s[POS22-1],
        "pos23": s[POS23-1],
        "pos24": s[POS24-1],
        "block20_24": block_20_24(s),
        "max_homopolymer": max_homopolymer(s),
        "buckets": ",".join(buckets),
        "predicted_direction": predicted,
        "rationale": rationale
    })

df_out = pd.DataFrame(rows)

# Order columns nicely
cols = ["sequence","GC_frac","pos20","pos21","pos22","pos23","pos24","block20_24",
        "max_homopolymer","buckets","predicted_direction","rationale"]
df_out = df_out[cols]

df_out.to_csv(OUT_CSV, index=False)
print(f"✓ Wrote {OUT_CSV}")
display(df_out)


✓ Wrote panel20_labeled.csv


,sequence,GC_frac,pos20,pos21,pos22,pos23,pos24,block20_24,max_homopolymer,buckets,predicted_direction,rationale
0,CCGGTTCCACTATATGTTG,0.473684,T,G,T,T,G,TGTTG,2,pos22=T,increase,pos22=T
1,GTTGATACGGTAAATGTGC,0.421053,T,G,T,G,C,TGTGC,3,pos22=T,increase,pos22=T
2,ACTTCCGGTCCTCGATTTA,0.473684,A,T,T,T,A,ATTTA,3,pos22=T,increase,pos22=T
3,AGCTATCTGTCCGTGGTGT,0.526316,G,G,T,G,T,GGTGT,2,"pos22=T,pos20=G",mixed,pos22=T; pos20=G
4,CCGCCGAGTATCGTAATCG,0.578947,A,A,T,C,G,AATCG,2,"pos22=T,pos21=A",increase,pos22=T; pos21=A
5,CTCTTAAAGACACGCAGCT,0.473684,C,A,G,C,T,CAGCT,3,"pos22=G,pos21=A",mixed,pos22=G; pos21=A
6,TCCGAGTACGTGCTTATGA,0.473684,T,A,T,G,A,TATGA,2,"pos22=T,pos21=A",increase,pos22=T; pos21=A
7,ACGCCAAGCCACGTAATAA,0.473684,A,A,T,A,A,AATAA,2,"pos22=T,pos21=A,AATAA@20–24",increase,pos22=T; pos21=A; AATAA@20–24
8,AGACACCGTCGGGCAAACT,0.578947,A,A,A,C,T,AAACT,3,pos21=A,increase,pos21=A
9,TGGTTGTGAGTTGCTAGTC,0.473684,T,A,G,T,C,TAGTC,2,"pos22=G,pos21=A",mixed,pos22=G; pos21=A


Write the fasta to imput it into the R pipline and then check out the sequnces expression predicted by the Model. That pipline was done 3 times to get for what i belife nice sequnces to construct in the lab.

In [24]:
import pandas as pd

INPUT_CSV = "outputs/panel20_candidates.csv"
OUTPUT_FASTA = "panel20_with_flanks.fasta"

PREFIX = "AAGG"
SUFFIX = "GCTT"

df = pd.read_csv(INPUT_CSV)

with open(OUTPUT_FASTA, "w") as f:
    for i, seq in enumerate(df["sequence"], start=1):
        core = str(seq).strip().upper()
        full = PREFIX + core + SUFFIX
        f.write(f">seq{i}\n{full}\n")

print(f"✓ FASTA written to {OUTPUT_FASTA}")


✓ FASTA written to panel20_with_flanks.fasta
